# ADBE Capital Allocation

The Economic Value and Compounding lenses asked *how good the business economics are*. This notebook asks a different owner question: **what did management do with the cash the business generated, and did those decisions help or hurt per-share owners?**

Everything is deterministic and built from the trusted Feature 1 free cash flow, diluted shares, net cash, and ROIC, the Feature 2A annual snapshots, and three new reported facts — repurchases, stock-based compensation, and dividends. The classification is a documented rule with named drivers, never an opaque score, and it summarizes *observable outcomes*, not management intent or valuation.

Three ideas do the heavy lifting:

1. **Buyback spending is not the same as share-count reduction.** A company can spend billions on buybacks and still see flat or rising shares if stock-based compensation and other issuance offset it. We judge buybacks by the *actual* diluted-share-count change.
2. **SBC is an owner cost, but it does not map one-for-one to dilution.** We show SBC and SBC/FCF as a burden signal, and we never subtract SBC from free cash flow.
3. **Distributions must be read against the balance sheet.** Drawing down surplus cash while staying strong is fine; funding returns by deepening net debt is not.

> Set an SEC User-Agent identifying your application and an administrative contact. Replace the sample contact below before making live requests.

In [ ]:
import os

from owner_lens import (
    SecClient,
    capital_allocation_from_facts,
    capital_allocation_summary,
    format_capital_allocation_view,
)

USER_AGENT = os.environ.get("OWNER_LENS_SEC_USER_AGENT", "OwnerLens admin@example.com")

raw = SecClient(USER_AGENT).retrieve_company_facts("ADBE").raw_facts
rows = capital_allocation_from_facts(raw)
[r.fiscal_year for r in rows]

## The compact per-year view

One row per fiscal year, newest first, in millions. `Repo/FCF` is repurchases as a share of free cash flow, `CR/FCF` is total capital returned over free cash flow (above 100% means Adobe returned more than it generated), `Retained` is free cash flow minus distributions (negative when returns exceed free cash flow), and `Buyback` is the effectiveness after dilution.

In [ ]:
print(format_capital_allocation_view(rows))

## Questions 1-4: cash generated, buybacks, dividends, and the SBC burden

Free cash flow is the cash available to return. Repurchases and dividends are what actually went out. SBC/FCF sizes the compensation burden that buybacks must overcome before owners see any share reduction.

In [ ]:
def m(value):
    return f"{value / 1e6:,.0f}" if value is not None else "n/a"


def pct(value):
    return f"{value * 100:.0f}%" if value is not None else "n/a"


for r in rows:
    print(f"FY{r.fiscal_year}")
    print(f"    free cash flow: {m(r.free_cash_flow)}")
    print(f"    repurchases:    {m(r.repurchases)}  ({pct(r.repurchases_over_fcf)} of FCF)")
    print(f"    dividends:      {'none' if r.dividends is None else m(r.dividends)}")
    print(f"    SBC:            {m(r.sbc)}  ({pct(r.sbc_over_fcf)} of FCF)")

## Questions 5-6: did shares actually shrink, and were buybacks effective?

This is the crux. Compare the money spent on buybacks with the real change in diluted shares. If shares fell meaningfully, the buybacks overcame SBC dilution; if shares stayed flat or rose despite heavy spending, they did not.

In [ ]:
for r in rows:
    print(
        f"FY{r.fiscal_year}: share change {pct(r.diluted_share_growth)}  "
        f"->  {r.buyback_effectiveness.value}"
    )

## Question 7: were distributions funded from surplus cash or worsening leverage?

Net cash or net debt is shown start-to-end by the row's trajectory. A falling cash balance that stays net-cash positive is a healthy drawdown of surplus; a turn to net debt or a deepening net-debt position is a real deterioration and is flagged as such.

In [ ]:
for r in rows:
    retained = m(r.retained_fcf)
    flag = "BALANCE_SHEET_DETERIORATED" if any(
        d.value == "BALANCE_SHEET_DETERIORATED" for d in r.drivers
    ) else "healthy"
    print(
        f"FY{r.fiscal_year}: net cash {m(r.net_cash_or_debt)}  "
        f"retained FCF {retained}  cap returned/FCF {pct(r.capital_returned_over_fcf)}  [{flag}]"
    )

## Question 8: did retained capital operate in a high-ROIC business?

ROIC is context only — it indicates whether retaining and reinvesting capital happens in an economically productive business. OwnerLens does not claim the retained dollars earned exactly this return, and it computes no return on retained free cash flow.

In [ ]:
for r in rows:
    codes = ", ".join(d.value for d in r.drivers) or "none"
    print(f"FY{r.fiscal_year}  ROIC {pct(r.roic)}  {r.classification.value}")
    print(f"    {codes}")

## Question 9: what does OwnerLens conclude?

The multi-year summary aggregates the owner questions into one view.

In [ ]:
print(capital_allocation_summary(rows))

Read the classifications and drivers together. Consistent effective buybacks that overcome a real SBC burden, a share count that actually falls, and a balance sheet that stays healthy while ROIC remains high is owner-friendly capital allocation — regardless of how large the distributions look.

What this lens is **not**: it makes no judgment about whether repurchases occurred below intrinsic value, and it involves no valuation, expected return, or total-shareholder-return claim. It reports what management did and what actually happened to per-share owners, with centralized, inspectable thresholds that are intentionally imprecise and not tuned to make Adobe look good.